# 🎮 Florr.io Mob 检测器训练 - YOLO26s

使用最新的 YOLO26s 模型训练 Florr.io 游戏中的怪物检测器。

## 📋 训练流程

1. ✅ 环境检查
2. ✅ 安装依赖
3. ✅ 生成训练数据
4. ✅ 训练模型
5. ✅ 验证模型
6. ✅ 下载模型

---

## 1️⃣ 环境检查

In [ ]:
import torch
import os

print("=" * 60)
print("环境信息")
print("=" * 60)
print(f"Python版本: {torch.__version__}")
print(f"CUDA可用: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"✓ GPU型号: {torch.cuda.get_device_name(0)}")
    print(f"✓ 显存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("✗ GPU不可用，请检查设置")
    print("提示: 右侧 Settings > Accelerator > GPU P100")

## 2️⃣ 安装依赖

In [ ]:
print("安装 Ultralytics...")
!pip install -q ultralytics
print("✓ Ultralytics 安装完成")

from ultralytics import YOLO
print("✓ 导入成功")

## 3️⃣ 配置数据集路径

**重要**: 修改 `DATASET_NAME` 为您上传的数据集名称

In [ ]:
# 🔧 配置参数
DATASET_NAME = 'florr-mob-training-package'  # 修改为您的数据集名称
NUM_SAMPLES = 5000  # 训练样本数量
EPOCHS = 100  # 训练轮数
BATCH_SIZE = 16  # Batch size（显存不足改为8）

# 自动设置路径
DATASET_PATH = f'/kaggle/input/{DATASET_NAME}'
WORK_PATH = '/kaggle/working'

print(f"数据集路径: {DATASET_PATH}")
print(f"工作路径: {WORK_PATH}")

# 检查数据集是否存在
if os.path.exists(DATASET_PATH):
    print("✓ 数据集已找到")
    
    # 列出数据集内容
    print("\n数据集内容:")
    for item in os.listdir(DATASET_PATH):
        item_path = os.path.join(DATASET_PATH, item)
        if os.path.isdir(item_path):
            count = len(os.listdir(item_path))
            print(f"  📁 {item}/ ({count} items)")
        else:
            size = os.path.getsize(item_path) / 1024 / 1024
            print(f"  📄 {item} ({size:.2f} MB)")
else:
    print(f"✗ 数据集未找到: {DATASET_PATH}")
    print("\n请先添加数据集:")
    print("1. 点击右侧 'Add Data'")
    print("2. 搜索您的数据集名称")
    print("3. 点击 'Add'")

## 4️⃣ 生成训练数据

In [ ]:
import sys
sys.path.append(f'{DATASET_PATH}/scripts')

from generate_data import HighQualityDataGenerator

print("=" * 60)
print("生成训练数据集")
print("=" * 60)

# 创建数据生成器
generator = HighQualityDataGenerator(
    mob_dir=f'{DATASET_PATH}/mob_images',
    bg_dir=f'{DATASET_PATH}/backgrounds',
    output_dir=f'{WORK_PATH}/dataset'
)

# 生成数据集
generator.generate_dataset(
    total_samples=NUM_SAMPLES,
    train_ratio=0.85,
    num_mobs_range=(5, 20),
    scale_range=(0.06, 0.5)
)

print("\n✓ 数据集生成完成")
print(f"训练集: {WORK_PATH}/dataset/train/")
print(f"验证集: {WORK_PATH}/dataset/val/")

## 5️⃣ 训练模型

In [ ]:
print("=" * 60)
print("训练 YOLO26s 模型")
print("=" * 60)

# 加载预训练模型
model_path = f'{DATASET_PATH}/yolo26s.pt'
if os.path.exists(model_path):
    print(f"加载模型: {model_path}")
    model = YOLO(model_path)
else:
    print("使用在线模型（会自动下载）")
    model = YOLO('yolo26s.pt')

print(f"\n训练参数:")
print(f"  - 样本数: {NUM_SAMPLES}")
print(f"  - Epochs: {EPOCHS}")
print(f"  - Batch size: {BATCH_SIZE}")
print(f"  - 图像尺寸: 640")

print("\n开始训练...")
print("=" * 60)

# 训练模型
results = model.train(
    data=f'{WORK_PATH}/dataset/data.yaml',
    epochs=EPOCHS,
    imgsz=640,
    batch=BATCH_SIZE,
    name='mob_detector_yolo26s',
    project=f'{WORK_PATH}/runs',
    device='cuda',
    patience=20,
    save=True,
    plots=True,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=3,
    workers=4,
    amp=True,
)

print("\n" + "=" * 60)
print("✓ 训练完成!")
print("=" * 60)

## 6️⃣ 验证模型

In [ ]:
print("=" * 60)
print("验证模型")
print("=" * 60)

# 验证模型
metrics = model.val()

print(f"\n验证结果:")
print(f"  mAP50: {metrics.box.map50:.4f}")
print(f"  mAP50-95: {metrics.box.map:.4f}")
print(f"  Precision: {metrics.box.mp:.4f}")
print(f"  Recall: {metrics.box.mr:.4f}")

print("\n✓ 验证完成")

## 7️⃣ 保存模型

In [ ]:
import shutil
from pathlib import Path

print("=" * 60)
print("保存模型")
print("=" * 60)

# 模型路径
best_model = Path(f'{WORK_PATH}/runs/mob_detector_yolo26s/weights/best.pt')
last_model = Path(f'{WORK_PATH}/runs/mob_detector_yolo26s/weights/last.pt')

print(f"\n最佳模型: {best_model}")
print(f"最后模型: {last_model}")

# 打包训练结果
print("\n打包训练结果...")
shutil.make_archive(
    f'{WORK_PATH}/mob_detector_results',
    'zip',
    f'{WORK_PATH}/runs/mob_detector_yolo26s'
)

print("✓ 打包完成: mob_detector_results.zip")

# 列出输出文件
print("\n输出文件:")
for item in os.listdir(WORK_PATH):
    item_path = os.path.join(WORK_PATH, item)
    if os.path.isfile(item_path):
        size = os.path.getsize(item_path) / 1024 / 1024
        print(f"  📄 {item} ({size:.2f} MB)")

## 8️⃣ 测试模型（可选）

In [ ]:
# 测试单张图片
test_images = list(Path(f'{WORK_PATH}/dataset/val/images').glob('*.jpg'))[:3]

if test_images:
    print(f"测试 {len(test_images)} 张图片...")
    
    for img_path in test_images:
        print(f"\n测试: {img_path.name}")
        results = model.predict(str(img_path), save=True)
        
        # 显示结果
        from IPython.display import Image, display
        display(Image(filename=results[0].save_dir / results[0].path.name))
else:
    print("未找到测试图片")

## 📥 下载模型

### 方法1: 从 Output 下载

1. 点击右侧 **Output** 标签
2. 找到以下文件:
   - `mob_detector_results.zip` - 完整训练结果
   - `runs/mob_detector_yolo26s/weights/best.pt` - 最佳模型
3. 点击文件右侧的下载图标

### 方法2: 直接下载

运行以下代码:

In [ ]:
# 显示下载链接（Kaggle 会自动提供下载按钮）
print("\n" + "=" * 60)
print("训练完成！")
print("=" * 60)
print("\n📥 下载模型:")
print("1. 点击右侧 'Output' 标签")
print("2. 下载以下文件:")
print("   - mob_detector_results.zip (完整结果)")
print("   - runs/mob_detector_yolo26s/weights/best.pt (最佳模型)")
print("\n🎮 使用模型:")
print("```python")
print("from ultralytics import YOLO")
print("model = YOLO('best.pt')")
print("results = model.predict('game_screenshot.jpg')")
print("results[0].show()")
print("```")
print("\n祝游戏愉快！🚀")